In [11]:
import os
import pandas as pd
import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                r'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', 
                SCOPES
            )
            creds = flow.run_local_server(port=0)
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def update_sheet(spreadsheetId, spread_range, scopes, creds, df):
    # Make a copy to avoid changing the original dataframe
    df_clean = df.copy()
    
    # 1. ALWAYS convert datetime to string FIRST before modifying fillna values
    for col in df_clean.columns:
        if pd.api.types.is_datetime64_any_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%d')
            
    # 2. Force conversion to base objects and completely clean all NaN/NaT/<NA> types
    df_clean = df_clean.astype(object).fillna('')
    
    # 3. Double-check element-by-element for any sneaky datetime objects missed by type matching
    header = df_clean.columns.tolist()
    data = df_clean.values.tolist()
    
    clean_data = []
    for row in data:
        clean_row = []
        for cell in row:
            if isinstance(cell, (datetime, pd.Timestamp)):
                clean_row.append(cell.strftime('%Y-%m-%d'))
            elif pd.isna(cell):
                clean_row.append('')
            else:
                clean_row.append(cell)
        clean_data.append(clean_row)
        
    values_with_header = [header] + clean_data
    
    try:
        service = build('sheets', 'v4', credentials=creds)
        service.spreadsheets().values().clear(spreadsheetId=spreadsheetId, range=spread_range, body={}).execute()
        
        body = {'values': values_with_header}
        result = service.spreadsheets().values().update(
            spreadsheetId=spreadsheetId, 
            range=spread_range, 
            valueInputOption="USER_ENTERED", 
            body=body
        ).execute()
        print(f"{result.get('updatedCells')} cells updated (sheet replaced).")
        return result
    except HttpError as error:
        print(f"An error occurred: {error}")
        return error
    

def get_excel_data(file_path, sheet_name, usecols, skiprows, nrows, header_row):
    """
    Custom helper function to read specific columns and rows from an Excel sheet.
    """
    try:
        df = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            usecols=usecols,
            skiprows=skiprows - 1,  # Adjusting for 0-indexed pandas row count
            nrows=nrows,
            header=header_row
        )
        return df
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None
    
def extract_data(file_path):
    # 1. Fetch raw data (including row index 0 which contains 'Plan' / 'Real')
    combined_df = get_excel_data(file_path, 'update 19 jan 26 (UPtes (2)', "A:M, BH:BI, BQ:CV", 2, 1000, 1)
    
    if combined_df is not None:
        # --- FIX STEP A: Rebuild headers using Row 0 ('Plan'/'Real') and Timestamps ---
        new_columns = []
        current_date = None
        
        # Row 0 contains the indicator 'Plan' or 'Real' (which stands for Realisasi)
        indicator_row = combined_df.iloc[0].fillna('').astype(str).str.strip().str.lower()
        
        for col_name, indicator in zip(combined_df.columns, indicator_row):
            col_str = str(col_name).strip()
            
            # If the column header looks like a date/timestamp, lock it in
            if '2026' in col_str and 'unnamed' not in col_str.lower():
                current_date = col_str.split()[0]  # Gets '2026-08-01'
            
            # Combine the date with the type indicator if it's a monthly target column
            if current_date and ('plan' in indicator or 'real' in indicator):
                clean_type = 'Plan' if 'plan' in indicator else 'Realisasi'
                new_columns.append(f"{current_date}_{clean_type}")
            else:
                new_columns.append(col_str.lower())
                
        combined_df.columns = new_columns
        
        # Drop row 0 now that its information is stored safely in the headers
        combined_df = combined_df.drop(combined_df.index[0]).reset_index(drop=True)
        
        # --- FIX STEP B: Setup Variable melting ---
        id_vars = ['no.', '3. no. project / wbs', '2. item capex', '4. entitas', '5. departemen', 'source_file']
        id_vars = [col for col in id_vars if col in combined_df.columns]
        
        # Target only our newly constructed date columns (e.g., '2026-08-01_Plan')
        value_vars = [col for col in combined_df.columns if '_plan' in col or '_realisasi' in col]
        
        if not value_vars:
            print("Warning: No structured monthly date columns found to melt.")
            return combined_df

        # 2. Melt wide months down to rows
        df_melted = pd.melt(
            combined_df,
            id_vars=id_vars,
            value_vars=value_vars,
            var_name='Metric_Raw',
            value_name='Value'
        )
        
        # 3. Extract the 'Type' and the 'Month'
        # Split '2026-08-01_Plan' into ['2026-08-01', 'Plan']
        df_melted[['Date_Raw', 'Type']] = df_melted['Metric_Raw'].str.split('_', expand=True)
        
        # --- FIX STEP C: Convert date string to the LAST day of that month (dd/mm/yyyy) ---
        def get_end_of_month(date_str):
            try:
                # Convert '2026-08-01' to datetime object
                dt = pd.to_datetime(date_str)
                # Roll forward to the last day of that month
                end_date = dt + pd.offsets.MonthEnd(0)
                return end_date.strftime('%d/%m/%Y')
            except:
                return 'Unknown'
                
        df_melted['Month'] = df_melted['Date_Raw'].apply(get_end_of_month)
        df_melted = df_melted[df_melted['Month'] != 'Unknown']
        
        # 4. Pivot Plan and Realisasi back into clean individual columns
        pivot_index = id_vars + ['Month']
        df_clean = df_melted.pivot_table(
            index=pivot_index,
            columns='Type',
            values='Value',
            aggfunc='first'
        ).reset_index()
        
        # Clean structural formatting rules
        for col in ['Plan', 'Realisasi']:
            if col not in df_clean.columns:
                df_clean[col] = None
        df_clean.columns.name = None
        
        print("\nSuccessfully restructured data into a long, clean framework!")
        return df_clean
    else:
        print("\nNo data was collected. Please verify your file paths.")
        return None

def main():
    combined_df = extract_data(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\PROFILE CAPEX RKAP 1565 (5.  Mar 2026)c-2-4 (CHECK).xlsx")
    if combined_df is not None:
        display(combined_df.head(10))
        creds = connect_to_sheet()
        update_sheet('1ckgGhsbbHGYfpqXtak_EG79ElY00bTvAADbNySRMa1s', 'Sheet1', None, creds, combined_df)

if __name__ == '__main__':
    main()

,no,project code / wbs,plant,semen / non semen,entitas,plant.1,nama capex,tahun capex,tipe capex (ko/pko/m/po/str),coc/new/multiyears,...,2026-08-01_Plan,2026-08-01_Realisasi,2026-09-01_Plan,2026-09-01_Realisasi,2026-10-01_Plan,2026-10-01_Realisasi,2026-11-01_Plan,2026-11-01_Realisasi,2026-12-01_Plan,2026-12-01_Realisasi
0,715.0,P6-22270-07-C-12,NaN,Semen,GhoPo,,"IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM...",2022.0,PO,MULTIYEARS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,711.0,P2-23193,TUBAN,Semen,GhoPo,TUBAN,GP-3 ADDITIONAL FIRE ALARM MSD 803TX3B,2023.0,KO,COC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,712.0,P2-23593,TUBAN,Semen,GhoPo,TUBAN,GP-J Develop OPT. KAP. DERMAGA GRESIK JF,2023.0,PKO,COC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,713.0,P6-23196,NaN,Semen,GhoPo,,GP-4 Additional storage GH 514bg02,2023.0,PO,MULTIYEARS,...,NaN,NaN,NaN,NaN,1604000000,NaN,NaN,NaN,NaN,NaN
4,660.0,P2-24104,TUBAN,Semen,GhoPo,TUBAN,GP-T RETROFIT GUDANG HANDAK QG 10000011,2024.0,M,COC,...,NaN,NaN,149170484.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,661.0,P2-24435,TUBAN,Semen,GhoPo,TUBAN,GP-2 REPAIR BOZEM WST 712BZ01,2024.0,M,COC,...,NaN,NaN,318700000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,662.0,P2-24434,TUBAN,Semen,GhoPo,TUBAN,GP-1 REPLACE CONE BOTTOM CT RM 341CT1,2024.0,KO,COC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,663.0,P2-24182,TUBAN,Semen,GhoPo,TUBAN,GP-2 ADDITIONAL BULK STATION CTB 613BK1,2024.0,KO,COC,...,NaN,NaN,NaN,NaN,264466700,NaN,NaN,NaN,NaN,NaN
8,664.0,P2-24030,TUBAN,Semen,GhoPo,TUBAN,GP-2 REPLACE CHAIN & BUCKET RECLAIMER RH,2024.0,KO,COC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,665.0,P2-24053,TUBAN,Semen,GhoPo,TUBAN,GP-1 REPAIR IMPELLER ID FAN RPT 341FN4,2024.0,KO,COC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


47000 cells updated (sheet replaced).
